# ATOM-SLM — test a checkpoint from the Hugging Face Hub

Loads a fine-tuned model, asks it math questions, and prints the raw generation
plus the parsed final answer. No training and no repo clone.

**Setup**

1. Settings → Accelerator: **GPU T4 x2** (CPU works, just slower). Internet: **On**.
2. Optional: Add-ons → Secrets → `HF_TOKEN` — only needed for private models.

The default is the Stage 1 model `Asaadzx/atom-slm-math-135m`. Change `MODEL_ID`
in section 1 to test any other checkpoint.

In [ ]:
import sys

import torch
import transformers

print("python      :", sys.version.split()[0])
print("torch       :", torch.__version__, "| cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu         :", torch.cuda.get_device_name(0),
          "| devices:", torch.cuda.device_count())
print("transformers:", transformers.__version__)

## 1 — Pick the model and the questions

`MODEL_ID` can be a Hub id or a local path (e.g. `/kaggle/working/runs/m1-...`).
`COMPARE_WITH_BASE = True` loads the base model too and prints both answers next
to each other — the fastest way to see what fine-tuning changed.

In [ ]:
MODEL_ID          = "Asaadzx/atom-slm-math-135m"
BASE_MODEL_ID     = "HuggingFaceTB/SmolLM2-135M"
MAX_NEW_TOKENS    = 384
COMPARE_WITH_BASE = False
DOWNLOAD_SNAPSHOT = False          # True -> copy the whole repo to /kaggle/working/models/
RESULTS_DIR       = "/kaggle/working/results"

PROMPTS = [
    "Natalia sold clips to 48 friends in April, and she sold half as many clips "
    "in May. How many clips did she sell altogether?",
    "A robe takes 2 bolts of blue fiber and half that much white fiber. "
    "How many bolts in total does it take?",
    "What is 17 * 24? Show the steps.",
]

print("model:", MODEL_ID)

In [ ]:
import os

try:
    from kaggle_secrets import UserSecretsClient

    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    from huggingface_hub import login

    login(os.environ["HF_TOKEN"])
except Exception as exc:  # public model, no secret attached, or not on Kaggle
    print("Skipping login:", type(exc).__name__, "- fine for public models.")

In [ ]:
if DOWNLOAD_SNAPSHOT:
    from huggingface_hub import snapshot_download

    local_dir = snapshot_download(MODEL_ID, local_dir=f"/kaggle/working/models/{MODEL_ID.split('/')[-1]}")
    print("Downloaded to:", local_dir)
else:
    print("DOWNLOAD_SNAPSHOT = False — the model is fetched into the HF cache on load.")

## 2 — Load the model and ask

The prompt layout matters. This model was trained without a chat template, so it
expects exactly:

```
### Question:
<question>

### Solution:
```

Chat models (`tokenizer.chat_template` set) automatically get their own template.

In [ ]:
import re

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


def load(model_id):
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    kwargs = {"attn_implementation": "sdpa"}
    try:                      # transformers v5
        model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float16, **kwargs)
    except TypeError:         # transformers v4
        model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, **kwargs)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    return tokenizer, model.to(device).eval(), device


def build_prompt(tokenizer, question):
    if getattr(tokenizer, "chat_template", None):
        return tokenizer.apply_chat_template(
            [{"role": "user", "content": question}],
            tokenize=False,
            add_generation_prompt=True,
        )
    return f"### Question:\n{question}\n\n### Solution:\n"


@torch.no_grad()
def ask(tokenizer, model, question, max_new_tokens=MAX_NEW_TOKENS):
    prompt = build_prompt(tokenizer, question)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )
    text = tokenizer.decode(output[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return text


def extract_answer(text):
    if "####" in text:
        tail = [line.strip() for line in text.split("####")[-1].splitlines() if line.strip()]
        return tail[0] if tail else None
    boxed = re.findall(r"\\boxed\{([^}]*)\}", text)
    if boxed:
        return boxed[-1].strip()
    matches = re.findall(r"(?:the\s+)?(?:final\s+)?answer\s+is\s*[:\-]?\s*(.+)", text, re.IGNORECASE)
    return matches[-1].strip() if matches else None


tokenizer, model, device = load(MODEL_ID)
print("loaded on:", device)
print("chat template:", bool(getattr(tokenizer, "chat_template", None)))

In [ ]:
for question in PROMPTS:
    text = ask(tokenizer, model, question)
    print("=" * 78)
    print("Q:", question)
    print("answer:", extract_answer(text))
    print("-" * 78)
    print(text[:1200])

## 3 — Quick GSM8K sanity (20 questions)

A full evaluation is in `atom_slm_kaggle_t4.ipynb` (section 8). This is the
2-minute version: 20 items, 0-shot, same prompt as training. Expect the Stage 1
checkpoint around **5–15%**; the base model is ~0–1%.

In [ ]:
import json
import time
from pathlib import Path

from datasets import load_dataset

N = 20
dataset = load_dataset("openai/gsm8k", "main", split="test").select(range(N))

correct = formatted = 0
records = []
started = time.time()
for row in dataset:
    text = ask(tokenizer, model, row["question"])
    gold = row["answer"].split("####")[-1].strip()
    prediction = extract_answer(text)
    is_correct = prediction is not None and prediction.replace(",", "").replace("$", "") == gold
    correct += int(is_correct)
    formatted += int(prediction is not None)
    records.append(
        {"question": row["question"], "gold": gold, "prediction": prediction,
         "correct": is_correct, "output": text}
    )

print(f"GSM8K {N}: accuracy {100 * correct / N:.1f}% | format {100 * formatted / N:.1f}% "
      f"| {time.time() - started:.0f}s")

Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)
report = {
    "model": MODEL_ID,
    "limit": N,
    "accuracy": correct / N,
    "format_rate": formatted / N,
    "predictions": records,
}
out = Path(RESULTS_DIR) / f"test-{MODEL_ID.split('/')[-1]}-{N}.json"
out.write_text(json.dumps(report, indent=2))
print("report:", out)

for record in records[:3]:
    print("-" * 78)
    print("gold:", record["gold"], "| pred:", record["prediction"], "| correct:", record["correct"])
    print(record["output"][:300].replace("\n", " "))

## 4 — Optional: compare with the base model

In [ ]:
if COMPARE_WITH_BASE:
    base_tokenizer, base_model, _ = load(BASE_MODEL_ID)
    for question in PROMPTS[:2]:
        print("=" * 78)
        print("Q:", question)
        print("\nBASE :\n", ask(base_tokenizer, base_model, question)[:900])
        print("\nTUNED:\n", ask(tokenizer, model, question)[:900])
else:
    print("COMPARE_WITH_BASE = False")

## Troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `OSError` / `401` on load | private repo, no token | attach `HF_TOKEN` as a Kaggle secret and rerun section 1 |
| Output is empty or one word | prompt does not end with `### Solution:` | keep `build_prompt` unchanged; this model has no chat template |
| Output loops (`the the the`) | wrong prompt, or an undertrained checkpoint | run `python -m atom_slm.diagnose --run <run_dir>` in the training repo |
| Answer is right but not `####` | early checkpoint | retrain longer; see `atomslm_dev_stage2.ipynb` |
| Very slow on CPU | fp16 ignored on CPU | use GPU T4, or reduce `MAX_NEW_TOKENS` |